In [11]:
# Experiment Setup: API & Models

from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")
print(api_key[:8], "…")
client = OpenAI(api_key=api_key)

models = client.models.list()

for m in models.data:
    print(m.id)

sk-proj- …
gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-4o-search-preview-2025-03-11
gpt-5.3-codex
gpt-realtime-1.5
gpt-audio-1.5
gpt-4o-search-preview
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-4-1106-preview
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-4-0125-preview
gpt-4-turbo-preview
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03

In [2]:
#Pure LLM - Prompt/main.py

!pip -q install openai scipy progressbar2

import pandas as pd
import numpy as np
from get_prompt import *
from openai import OpenAI
from scipy.stats import fisher_exact
import progressbar
import os
from concurrent.futures import ThreadPoolExecutor
import time
from google.colab import userdata


DEBUG = False

# to access OpenAI API in China, open VPN first and then set the proxy

# run the following code in the terminal use python3

# load dataset
all_train = pd.read_csv("/content/LoRA_CTR_train.csv")

test_on_test = True # if True, test on test data, otherwise test on calibration data
if test_on_test:
    all_test = pd.read_csv("/content/LoRA_CTR_test.csv")
    result_path = '/content/results/result_gpt_test.npy'
else:
    all_test = pd.read_csv("/content/LoRA_CTR_calibration.csv")
    result_path = '/content/results/result_gpt_calibration.npy'


# if folder does not exist, create it
if not os.path.exists(os.path.dirname(result_path)):
    os.makedirs(os.path.dirname(result_path))


# convert pd dataframe to dict
def df_to_dict(df):
    test_id_ls = df["test_id"].unique()
    dict_df = {}
    for test_id in test_id_ls:
        dict_df[test_id] = df[df["test_id"] == test_id]
    return dict_df


all_train = df_to_dict(all_train)
all_test = df_to_dict(all_test)
if DEBUG:
    all_test = {k: all_test[k] for k in list(all_test)[:20]}



# split test data into significant and insignificant
# for each news in the test data, run significance test
def significance_test_one_news(impressions, clicks, CTRs):
    """
    Run significance test for one news. We compare the headline with the highest CTR with the rest of the headlines. If all such pairs are significant, we return True, otherwise False.

    :param impressions: list of impressions for each headline
    :param clicks: list of clicks for each headline
    :param CTRs: list of CTRs for each headline
    :return: True if all pairs are significant, False otherwise
    """

    # find the headline with the highest CTR.
    highest_CTR_index = CTRs.index(max(CTRs))

    # run pairwise significance test between the headline with the highest CTR and the rest of the headlines
    for i in range(len(CTRs)):
        if i == highest_CTR_index:
            continue
        pair_clicks = [clicks[highest_CTR_index], clicks[i]]
        pair_non_clicks = [
            impressions[highest_CTR_index] - clicks[highest_CTR_index],
            impressions[i] - clicks[i],
        ]
        contingency_table = [pair_clicks, pair_non_clicks]
        odds_ratio, p_value = fisher_exact(contingency_table)
        if p_value < 0.05:  # this pair is significant
            continue
        else:
            return False
    return True


significant_test = {}
significant_mask = []
# insignificant_test = {}
print("Running significance test for each news in the test data...")
for test_id in all_test.keys():
    one_news = all_test[test_id]
    impressions = one_news["impressions"].values.tolist()
    clicks = one_news["clicks"].values.tolist()
    CTRs = one_news["CTR"].values.tolist()
    significant = significance_test_one_news(impressions, clicks, CTRs)
    if significant:
        significant_test[test_id] = one_news
        significant_mask.append(True)
    else:
        significant_mask.append(False)
print("Significance test done!")
print('Among all {} test data, {} are significant.'.format(len(all_test), len(significant_test)))

# save significant_mask
result_dir = os.path.dirname(result_path)
np.save(os.path.join(result_dir, 'significant_mask.npy'), significant_mask)

n_headline = 0
n_correct_expected = 0
for test_id in significant_test.keys():
    one_news = significant_test[test_id]
    n_headline += len(one_news)
    n_correct_expected += 1/len(one_news)
print('Average number of headlines per news for significant news: ', n_headline / len(significant_test))
print('Random guess accuracy on significant test data: ', n_correct_expected/len(significant_test))

n_headline = 0
n_correct_expected = 0
for test_id in all_test.keys():
    one_news = all_test[test_id]
    n_headline += len(one_news)
    n_correct_expected += 1/len(one_news)
print('Average number of headlines per news for all test data: ', n_headline / len(all_test))
print('Random guess accuracy on all test data: ', n_correct_expected/len(all_test))

# run Prompt Based Method on both significant and insignificant test data

if test_on_test:
    test_combination = [
        # model name, is_flip, n_demo, test_sig
        # is_flip: whether we give the correct answer to the model or not. 0 for correct, 1 for incorrect
        #["gpt-3.5-turbo", 0, 0],  # zero-shot learning with significant samples
        #["gpt-3.5-turbo", 0, 2],  # in-context learning
        #["gpt-3.5-turbo", 1, 2],
        #["gpt-3.5-turbo", 0, 5],
        #["gpt-3.5-turbo", 1, 5],
        #["gpt-4-turbo", 0, 0],
        #["gpt-4-turbo", 0, 2],
        #["gpt-4-turbo", 1, 2],
        #["gpt-4-turbo", 0, 5],
        #["gpt-4-turbo", 1, 5],
        #["gpt-4o-2024-08-06", 0, 0],
        #["gpt-4o-2024-08-06", 0, 2],
        #["gpt-4o-2024-08-06", 1, 2],
        #["gpt-4o-2024-08-06", 0, 5],
        #["gpt-4o-2024-08-06", 1, 5],
        ["gpt-5.2", 0, 0],
        ["gpt-5.2", 0, 2],
        ["gpt-5.2", 1, 2],
        #["gpt-5.2-pro", 0, 5],
        #["gpt-5.2-pro", 1, 5]
    ]
else:
    test_combination = [
        # model name, is_flip, n_demo
        # is_flip: whether we give the correct answer to the model or not. 0 for correct, 1 for incorrect
        ["gpt-4-turbo", 0, 5],
    ]

# need API key to acess GPT
api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def predict_parallel(is_flip, model_name, n_demo):
    true_labels, pred_labels = [], []

    def process_one_news(one_news):
        if n_demo != 0:
            gpt_input, true_label = get_prompt_with_examples(demo_examples, one_news, is_flip)
        else:
            gpt_input, true_label = get_prompt_without_example(one_news)

        while True:
            try:
                completion = client.chat.completions.create(
                    model=model_name, messages=gpt_input, temperature=0.0
                )
                break
            except:
                time.sleep(1)
        pred_label = completion.choices[0].message.content
        return true_label, pred_label

    with ThreadPoolExecutor(max_workers=2) as executor:
        results = list(progressbar.progressbar(executor.map(process_one_news, all_test.values()), max_value=len(all_test)))

    for true_label, pred_label in results:
        true_labels.append(true_label)
        pred_labels.append(pred_label)

    return true_labels, pred_labels



# load previous results
try:
    result_gpt = list(np.load(result_path, allow_pickle=True))
except:
    result_gpt = []

for model_name, is_flip, n_demo in test_combination:
    print("--------------------------")
    print("Parameter setting: {} n_demo {} is_flip {}".format(model_name, n_demo, is_flip))

    # check if this combination has been tested before
    tested_before = False
    for result in result_gpt:
        if result['model_name'] == model_name and result['is_flip'] == is_flip and result['n_demo'] == n_demo:
            print("This combination has been tested before.")
            # print('Model name: {}, is_flip: {}, n_demo: {}'.format(result['model_name'], result['is_flip'], result['n_demo']))
            print("Accuracy on all:{:.2%}\t Accuracy on significant subset: {:.2%}".format(result['acc'], result['acc_sig']))
            tested_before = True

    if tested_before:
        continue


    pred_label = []

    # if test_sig == 0:
    #     test_samples = all_test
    #     true_label = all_train
    # elif test_sig == 1:
    #     test_samples = significant_test
    #     true_label = all_train

    if n_demo > 0:
        # sample n_demo samples from the training data (all_train)
        train_ids = list(all_train.keys())
        np.random.seed(12) # ensure for different test_combination, we have the same demo samples
        demo_ids = np.random.choice(train_ids, n_demo, replace=False)
        demo_examples = []
        for demo_id in demo_ids:
            one_sample = all_train[demo_id]
            demo_examples.append(one_sample)

    # for each news in test_samples, ask gpt to select the best headline
    true_label, pred_label = predict_parallel(is_flip, model_name, n_demo)
    whether_correct = [1 if x == y else 0 for x, y in zip(true_label, pred_label)]
    n_correct_all = sum(whether_correct)
    accur_all = n_correct_all / len(all_test)
    n_correct_sig = np.array(whether_correct)[significant_mask].sum()
    accur_sig = n_correct_sig / len(significant_test)

    print("Accuracy on all data: {:.2%}".format(accur_all))
    print("Accuracy on significant subset: {:.2%}".format(accur_sig))
    result_gpt.append({'model_name': model_name,
                        'is_flip': is_flip,
                        'n_demo': n_demo,
                        'pred_label': pred_label,
                        'true_label': true_label,
                        'acc': accur_all,
                        'acc_sig': accur_sig
                        })
    # save result_gpt
    np.save(result_path, result_gpt)


# result_gpt_df.to_csv("Pure LLM - Prompt/prompt-results/result_gpt_prompt.csv", index=False)

Running significance test for each news in the test data...
Significance test done!
Among all 3263 test data, 614 are significant.
Average number of headlines per news for significant news:  2.728013029315961
Random guess accuracy on significant test data:  0.536881495269117
Average number of headlines per news for all test data:  3.689549494330371
Random guess accuracy on all test data:  0.3301835393468923
--------------------------
Parameter setting: gpt-5.2 n_demo 0 is_flip 0


100% (3263 of 3263) |####################| Elapsed Time: 0:18:56 Time:  0:18:56


Accuracy on all data: 40.82%
Accuracy on significant subset: 67.92%
--------------------------
Parameter setting: gpt-5.2 n_demo 2 is_flip 0


100% (3263 of 3263) |####################| Elapsed Time: 0:16:43 Time:  0:16:43


Accuracy on all data: 41.62%
Accuracy on significant subset: 67.26%
--------------------------
Parameter setting: gpt-5.2 n_demo 2 is_flip 1


100% (3263 of 3263) |####################| Elapsed Time: 0:17:15 Time:  0:17:15


Accuracy on all data: 40.73%
Accuracy on significant subset: 68.08%


In [3]:
#Pure LLM - Prompt/visualize_result.py

!pip -q install seaborn

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
import os


result_dir = '/content/results'

result_gpt = list(np.load(os.path.join(result_dir, 'result_gpt_test.npy'), allow_pickle=True))
significant_mask = np.load(os.path.join(result_dir, 'significant_mask.npy'), allow_pickle=True)
CI_FOR_SIGNIFICANT = False # if True, calculate CI for significant news only, otherwise calculate CI for all news

def bootstrap_ci(data, n_bootstrap=1000, ci=95, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    bootstrap_samples = np.random.choice(data, (n_bootstrap, len(data)), replace=True)
    bootstrap_means = np.mean(bootstrap_samples, axis=1)
    lower_bound = np.percentile(bootstrap_means, (100-ci)/2)
    upper_bound = np.percentile(bootstrap_means, 100 - (100-ci)/2)
    return lower_bound, upper_bound

# Apply bootstrap on acc_vec for each row and update ci_lower and ci_upper
for experiment in result_gpt:
    correct = [1 if x == y else 0 for x, y in zip(experiment['true_label'], experiment['pred_label'])]
    if CI_FOR_SIGNIFICANT:
        correct = np.array(correct)[significant_mask]
    lower, upper = bootstrap_ci(correct, random_state=42)
    # result_gpt_df.at[i, 'ci_lower_bs'] = lower
    # result_gpt_df.at[i, 'ci_upper_bs'] = upper
    experiment['ci_lower_bs'] = lower
    experiment['ci_upper_bs'] = upper

# Show CI for each model in a more readable format
print("Model Name".ljust(20), "CI Lower Bound".ljust(15), "CI Upper Bound".ljust(15))
print("-" * 50)
for experiment in result_gpt:
    model_name = experiment['model_name']
    ci_lower = experiment['ci_lower_bs']
    ci_upper = experiment['ci_upper_bs']
    print(f"{model_name.ljust(20)} {str(ci_lower).ljust(20)} {str(ci_upper).ljust(20)}")


model_list = [
    f"{e['model_name']}-demo{e['n_demo']}-flip{e['is_flip']}"
    for e in result_gpt
]

# mean_list = np.array(result_gpt_df['acc'])
mean_list = [experiment['acc'] for experiment in result_gpt]
mean_diff_matrix = np.zeros((len(model_list) + 1, len(model_list) + 1))
pval_matrix = np.zeros((len(model_list) + 1, len(model_list) + 1))

# Append "random guess" model to the list
model_list.append("Random Guess")


for i in range(len(model_list) - 1):  # Exclude "random guess" in this loop
    for j in range(len(model_list) - 1):  # Exclude "random guess" in this loop
        rvs1 = [1 if x == y else 0 for x, y in zip(result_gpt[i]['true_label'], result_gpt[i]['pred_label'])]
        rvs2 = [1 if x == y else 0 for x, y in zip(result_gpt[j]['true_label'], result_gpt[j]['pred_label'])]
        if CI_FOR_SIGNIFICANT:
            rvs1 = np.array(rvs1)[significant_mask]
            rvs2 = np.array(rvs2)[significant_mask]
        # rvs1 = (np.array(result_gpt_df['pred_label'])[i] == true_label_sig).astype(int)
        # rvs2 = (np.array(result_gpt_df['pred_label'])[j] == true_label_sig).astype(int)
        stat, pval = stats.ttest_rel(rvs1, rvs2)

        # Store mean difference and p-value
        mean_diff_matrix[i, j] = (np.mean(rvs1) - np.mean(rvs2)) * 100
        pval_matrix[i, j] = pval

test_df = pd.read_csv("/content/LoRA_CTR_test.csv")
n_per_news = test_df.groupby("test_id").size().values
random_guess = float(np.mean(1.0 / n_per_news))
print("Random guess baseline:", random_guess)

# Compare each model with "random guess"
for i in range(len(model_list) - 1):
    # rvs = (np.array(result_gpt_df['pred_label'])[i] == true_label_sig).astype(int)
    rvs = [1 if x == y else 0 for x, y in zip(result_gpt[i]['true_label'], result_gpt[i]['pred_label'])]
    if CI_FOR_SIGNIFICANT:
        rvs = np.array(rvs)[significant_mask]
    mean_diff = (np.mean(rvs) - random_guess) * 100
    stat, pval = stats.ttest_1samp(rvs, random_guess)

    # Store mean difference and p-value
    mean_diff_matrix[i, len(model_list) - 1] = mean_diff
    mean_diff_matrix[len(model_list) - 1, i] = -mean_diff
    pval_matrix[i, len(model_list) - 1] = pval
    pval_matrix[len(model_list) - 1, i] = pval

# Store the "random guess" model's self-comparison (diagonal)
mean_diff_matrix[len(model_list) - 1, len(model_list) - 1] = 0
pval_matrix[len(model_list) - 1, len(model_list) - 1] = 1

# Move "random guess" to the first place
model_list = ["Random Guess"] + model_list[:-1]
mean_diff_matrix = np.vstack([mean_diff_matrix[-1], mean_diff_matrix[:-1]])
mean_diff_matrix = np.hstack([mean_diff_matrix[:, -1].reshape(-1, 1), mean_diff_matrix[:, :-1]])
pval_matrix = np.vstack([pval_matrix[-1], pval_matrix[:-1]])
pval_matrix = np.hstack([pval_matrix[:, -1].reshape(-1, 1), pval_matrix[:, :-1]])

mean_diff_df = pd.DataFrame(mean_diff_matrix, index=model_list, columns=model_list)
pval_df = pd.DataFrame(pval_matrix, index=model_list, columns=model_list)

mask_pval = pval_df >= 0.05

# Create a mask to hide the upper triangle and the diagonal
upper_triangle_mask = np.triu(np.ones_like(mean_diff_matrix, dtype=bool))
np.fill_diagonal(upper_triangle_mask, True)

# Combined mask to hide upper triangle and non-significant p-values
combined_mask = mask_pval | upper_triangle_mask

# Create a cross-style mask for p-values greater than or equal to 0.05
cross_mask = np.zeros_like(mean_diff_matrix, dtype=bool)
cross_mask[mask_pval & ~upper_triangle_mask] = True

# Plot the heatmap of mean differences with the combined mask applied
plt.figure(figsize=(12, 10))
ax = sns.heatmap(mean_diff_df, mask=upper_triangle_mask, annot=True, fmt=".2f", cmap="coolwarm", cbar_kws={'label': 'Mean Difference (%)'},
                  linewidths=.5, linecolor='white', vmin=-15, vmax=15, center=0)

# Apply cross-style mask
for i in range(len(model_list)):
    for j in range(len(model_list)):
        if cross_mask[i, j]:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='white', lw=1, hatch='\\'))

plt.title('Mean Differences in Accuracy (%)')
plt.xlabel('Model')
plt.ylabel('Model')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.savefig(os.path.join(result_dir, 'mean_differences_heatmap_multiple.pdf'), format='pdf', bbox_inches='tight')
plt.close()

# Create a mask to hide the upper triangle for the p-value heatmap
lower_triangle_mask = np.tril(np.ones_like(pval_df, dtype=bool), -1)

# Plot the heatmap of p-values with the lower triangle mask applied
plt.figure(figsize=(12, 10))
sns.heatmap(pval_df, mask=~lower_triangle_mask, annot=True, fmt=".3f", cmap="viridis", cbar_kws={'label': 'P-value'}, linewidths=.5, linecolor='white')
plt.title('P-values')
plt.xlabel('Model')
plt.ylabel('Model')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.savefig(os.path.join(result_dir, 'p_values_heatmap_multiple.pdf'), format='pdf', bbox_inches='tight')
plt.close()

Model Name           CI Lower Bound  CI Upper Bound 
--------------------------------------------------
gpt-3.5-turbo        0.2865384615384616   0.31720042905301865 
gpt-3.5-turbo        0.3294437634079069   0.3631703953417101  
gpt-5.2              0.3910435182347533   0.42568954949433035 
gpt-5.2              0.3999387067116151   0.4333435488813975  
gpt-5.2              0.38889825314128107  0.4232301562978854  
Random guess baseline: 0.330183539346886
